# CSF Gene Matrix - Exploratory Data Analysis

This notebook analyzes the sparse CSF gene matrix dataset before running FFSI experiments.

**Key questions:**
1. What are the dataset dimensions (samples × features)?
2. How sparse is the data? (percentage of zeros)
3. What is the distribution of non-zero values per feature and per sample?
4. What labels are available and what's their distribution?
5. Are there any constant features (all same value) that should be excluded?
6. What's the information gain distribution across features?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load the Data

In [ ]:
# Update this path to match your local setup
DATA_PATH = Path('/Users/macbook/Documents/FFSI-paper/data/CSF_final_gene_matrix_for_FFSI.pkl')

# Load the pickle file
df = pd.read_pickle(DATA_PATH)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape[0]:,} samples × {df.shape[1]:,} columns")

In [ ]:
# Basic info
print("\n" + "="*60)
print("BASIC INFO")
print("="*60)
print(f"\nDataFrame shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColumn dtypes:")
print(df.dtypes.value_counts())

In [ ]:
# Preview data
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Column names
print(f"\nFirst 20 column names:")
print(list(df.columns[:20]))
print(f"\nLast 10 column names:")
print(list(df.columns[-10:]))

## 2. Identify Label Column(s)

In [ ]:
# Look for potential label columns (non-numeric or low cardinality)
print("\n" + "="*60)
print("POTENTIAL LABEL COLUMNS")
print("="*60)

potential_labels = []

for col in df.columns:
    n_unique = df[col].nunique()
    dtype = df[col].dtype
    
    # Likely label if: categorical, object, or low unique count
    if dtype == 'object' or dtype.name == 'category' or n_unique <= 20:
        potential_labels.append({
            'column': col,
            'dtype': str(dtype),
            'n_unique': n_unique,
            'sample_values': df[col].value_counts().head(5).to_dict()
        })

if potential_labels:
    for info in potential_labels:
        print(f"\nColumn: '{info['column']}'")
        print(f"  dtype: {info['dtype']}")
        print(f"  unique values: {info['n_unique']}")
        print(f"  value counts: {info['sample_values']}")
else:
    print("No obvious label columns found. All columns appear to be high-cardinality numeric.")
    print("\nChecking unique value counts for ALL columns...")
    unique_counts = df.nunique().sort_values()
    print(unique_counts.head(20))

In [ ]:
# Manual check - uncomment and modify if you know the label column name
# LABEL_COLUMN = 'your_label_column_name'
# print(f"Label distribution for '{LABEL_COLUMN}':")
# print(df[LABEL_COLUMN].value_counts())

## 3. Sparsity Analysis

In [ ]:
# Identify feature columns (exclude potential labels)
# Modify this list based on what you found above
label_cols = []  # Add your label column names here, e.g., ['label', 'Stage']
feature_cols = [col for col in df.columns if col not in label_cols]

print(f"Number of feature columns: {len(feature_cols):,}")

# Get feature data only
features_df = df[feature_cols]

In [ ]:
# Overall sparsity (assuming 0 = sparse/missing)
print("\n" + "="*60)
print("SPARSITY ANALYSIS")
print("="*60)

total_elements = features_df.size
zero_count = (features_df == 0).sum().sum()
null_count = features_df.isnull().sum().sum()

print(f"\nTotal elements: {total_elements:,}")
print(f"Zero values: {zero_count:,} ({100*zero_count/total_elements:.2f}%)")
print(f"Null/NaN values: {null_count:,} ({100*null_count/total_elements:.4f}%)")
print(f"\nEffective sparsity (zeros + nulls): {100*(zero_count+null_count)/total_elements:.2f}%")

In [ ]:
# Sparsity per feature (column)
feature_zero_pct = (features_df == 0).sum() / len(features_df) * 100

print("\n" + "="*60)
print("SPARSITY PER FEATURE")
print("="*60)
print(f"\nMin sparsity: {feature_zero_pct.min():.2f}%")
print(f"Max sparsity: {feature_zero_pct.max():.2f}%")
print(f"Mean sparsity: {feature_zero_pct.mean():.2f}%")
print(f"Median sparsity: {feature_zero_pct.median():.2f}%")

# Count features by sparsity level
print(f"\nFeatures with >99% zeros: {(feature_zero_pct > 99).sum():,}")
print(f"Features with >95% zeros: {(feature_zero_pct > 95).sum():,}")
print(f"Features with >90% zeros: {(feature_zero_pct > 90).sum():,}")
print(f"Features with >50% zeros: {(feature_zero_pct > 50).sum():,}")
print(f"Features with <10% zeros: {(feature_zero_pct < 10).sum():,}")

In [ ]:
# Visualize sparsity distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of feature sparsity
axes[0].hist(feature_zero_pct, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(feature_zero_pct.median(), color='red', linestyle='--', label=f'Median: {feature_zero_pct.median():.1f}%')
axes[0].set_xlabel('Percentage of Zero Values')
axes[0].set_ylabel('Number of Features')
axes[0].set_title('Distribution of Feature Sparsity')
axes[0].legend()

# Cumulative distribution
sorted_sparsity = np.sort(feature_zero_pct)
cumulative = np.arange(1, len(sorted_sparsity) + 1) / len(sorted_sparsity) * 100
axes[1].plot(sorted_sparsity, cumulative)
axes[1].axhline(50, color='red', linestyle='--', alpha=0.5)
axes[1].axvline(feature_zero_pct.median(), color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Percentage of Zero Values')
axes[1].set_ylabel('Cumulative % of Features')
axes[1].set_title('Cumulative Distribution of Feature Sparsity')

plt.tight_layout()
plt.show()

In [ ]:
# Sparsity per sample (row)
sample_zero_pct = (features_df == 0).sum(axis=1) / len(feature_cols) * 100

print("\n" + "="*60)
print("SPARSITY PER SAMPLE")
print("="*60)
print(f"\nMin sparsity: {sample_zero_pct.min():.2f}%")
print(f"Max sparsity: {sample_zero_pct.max():.2f}%")
print(f"Mean sparsity: {sample_zero_pct.mean():.2f}%")
print(f"Median sparsity: {sample_zero_pct.median():.2f}%")

In [ ]:
# Visualize sample sparsity
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sample_zero_pct, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(sample_zero_pct.median(), color='red', linestyle='--', label=f'Median: {sample_zero_pct.median():.1f}%')
ax.set_xlabel('Percentage of Zero Values')
ax.set_ylabel('Number of Samples')
ax.set_title('Distribution of Sample Sparsity')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Feature Value Analysis

In [ ]:
# Check if features are binary (0/1) or continuous
print("\n" + "="*60)
print("FEATURE VALUE ANALYSIS")
print("="*60)

# Count unique values per feature
unique_counts = features_df.nunique()

print(f"\nUnique values per feature:")
print(f"  Min: {unique_counts.min()}")
print(f"  Max: {unique_counts.max()}")
print(f"  Mean: {unique_counts.mean():.1f}")
print(f"  Median: {unique_counts.median():.1f}")

# Count binary features (exactly 2 unique values)
binary_features = (unique_counts == 2).sum()
constant_features = (unique_counts == 1).sum()

print(f"\nConstant features (1 unique value): {constant_features:,}")
print(f"Binary features (2 unique values): {binary_features:,}")
print(f"Multi-value features (>2 unique values): {(unique_counts > 2).sum():,}")

In [ ]:
# If features are binary, check what values they take
if binary_features > 0:
    print("\nSample binary feature values:")
    binary_cols = unique_counts[unique_counts == 2].index[:5]
    for col in binary_cols:
        print(f"  {col}: {sorted(features_df[col].unique())}")

In [ ]:
# If features are continuous, show distribution statistics
if unique_counts.max() > 10:
    print("\nFeature value statistics (non-zero values only):")
    
    # Get statistics for non-zero values
    non_zero_stats = []
    for col in feature_cols[:100]:  # Sample first 100 features
        non_zero = features_df[col][features_df[col] != 0]
        if len(non_zero) > 0:
            non_zero_stats.append({
                'min': non_zero.min(),
                'max': non_zero.max(),
                'mean': non_zero.mean(),
                'std': non_zero.std()
            })
    
    if non_zero_stats:
        stats_df = pd.DataFrame(non_zero_stats)
        print(stats_df.describe())

In [ ]:
# Distribution of unique value counts
fig, ax = plt.subplots(figsize=(10, 5))

# Use log scale if range is large
if unique_counts.max() > 100:
    ax.hist(unique_counts, bins=50, edgecolor='black', alpha=0.7, log=True)
    ax.set_ylabel('Number of Features (log scale)')
else:
    ax.hist(unique_counts, bins=range(1, unique_counts.max() + 2), edgecolor='black', alpha=0.7)
    ax.set_ylabel('Number of Features')

ax.set_xlabel('Number of Unique Values')
ax.set_title('Distribution of Unique Value Counts per Feature')
plt.tight_layout()
plt.show()

## 5. Constant and Near-Constant Features

In [ ]:
# Features to potentially exclude from FFSI analysis
print("\n" + "="*60)
print("FEATURES TO CONSIDER EXCLUDING")
print("="*60)

# Constant features (provide no information)
constant_cols = unique_counts[unique_counts == 1].index.tolist()
print(f"\nConstant features (must exclude): {len(constant_cols)}")
if len(constant_cols) <= 10:
    print(f"  {constant_cols}")

# Near-constant features (>99% same value)
near_constant = []
for col in feature_cols:
    mode_pct = features_df[col].value_counts(normalize=True).iloc[0] * 100
    if mode_pct > 99 and col not in constant_cols:
        near_constant.append((col, mode_pct))

print(f"\nNear-constant features (>99% same value): {len(near_constant)}")
if len(near_constant) <= 10:
    for col, pct in near_constant:
        print(f"  {col}: {pct:.2f}% same value")

In [ ]:
# Summary of usable features
usable_features = [col for col in feature_cols if col not in constant_cols]

print(f"\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"\nTotal columns: {len(df.columns):,}")
print(f"Label columns: {len(label_cols)}")
print(f"Feature columns: {len(feature_cols):,}")
print(f"Constant features (to exclude): {len(constant_cols):,}")
print(f"Usable features for FFSI: {len(usable_features):,}")

## 6. Information Gain Analysis (if binary features)

In [ ]:
# Only run this section after identifying the label column
# Uncomment and modify LABEL_COLUMN

# LABEL_COLUMN = 'your_label_column'  # <-- SET THIS

# def entropy(y):
#     """Calculate entropy of a label array."""
#     if len(y) == 0:
#         return 0
#     probs = y.value_counts(normalize=True)
#     return -np.sum(probs * np.log2(probs + 1e-10))

# def information_gain(df, feature, label):
#     """Calculate information gain of a feature."""
#     total_entropy = entropy(df[label])
#     
#     # Weighted average entropy after split
#     weighted_entropy = 0
#     for value in df[feature].unique():
#         subset = df[df[feature] == value][label]
#         weight = len(subset) / len(df)
#         weighted_entropy += weight * entropy(subset)
#     
#     return total_entropy - weighted_entropy

# # Calculate IG for a sample of features (full calculation can be slow)
# print("Calculating information gain for sample of features...")
# sample_features = usable_features[:500]  # Adjust as needed
# 
# ig_values = {}
# for feat in sample_features:
#     ig_values[feat] = information_gain(df, feat, LABEL_COLUMN)
# 
# ig_series = pd.Series(ig_values).sort_values(ascending=False)
# 
# print(f"\nTop 20 features by Information Gain:")
# print(ig_series.head(20))
# 
# print(f"\nFeatures with zero IG: {(ig_series == 0).sum()} / {len(ig_series)}")

## 7. Recommendations for FFSI Experiments

In [ ]:
print("\n" + "="*60)
print("RECOMMENDATIONS FOR FFSI EXPERIMENTS")
print("="*60)

print("""
Based on the analysis above, consider the following before running FFSI:

1. LABEL COLUMN:
   - Identify which column(s) to use as labels
   - Update the experiment script with the correct column name

2. FEATURE FILTERING:
   - Exclude constant features (they provide no information)
   - Consider excluding near-constant features (>99% same value)
   - Consider minimum non-zero threshold for sparse features

3. BINARIZATION (if features are not already binary):
   - Gene expression values may need binarization
   - Common approaches: median split, threshold at 0, quantile-based

4. SAMPLE SIZES:
   - Given sparsity, larger samples may be needed
   - Suggested: 10k, 50k, 100k, 500k

5. FEATURE SUBSET SIZES:
   - Standard: 7, 8, 9, 10 features
   - May need adjustment based on total usable features
""")

In [ ]:
# Save summary statistics for reference
summary = {
    'total_samples': len(df),
    'total_columns': len(df.columns),
    'feature_columns': len(feature_cols),
    'constant_features': len(constant_cols),
    'usable_features': len(usable_features),
    'overall_sparsity_pct': 100 * (zero_count + null_count) / total_elements,
    'median_feature_sparsity_pct': feature_zero_pct.median(),
    'median_sample_sparsity_pct': sample_zero_pct.median(),
    'binary_features': binary_features,
    'data_path': str(DATA_PATH)
}

print("\nDataset Summary:")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")